In [ ]:




import cv2
import csv
import time


In [ ]:
# Open webcam to grab one frame for selection

cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

# Select multiple ROIs
bboxes = []
while True:
    bbox = cv2.selectROI("Select Object", frame, fromCenter=False, showCrosshair=True)
    if bbox == (0,0,0,0):
        break
    bboxes.append(bbox)

cv2.destroyWindow("Select Object")

In [ ]:
# Create individual CSRT trackers
trackers = []
object_id = 0
for bbox in bboxes:
    tracker = cv2.legacy.TrackerCSRT_create()
    tracker.init(frame, bbox)
    trackers.append((tracker, object_id))
    object_id += 1

# Prepare CSV file
csv_file = "tracking_log.csv"
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["object_id", "frame_number", "timestamp", "x", "y", "w", "h"])

In [ ]:
# Setup video writer
fourcc = cv2.VideoWriter_fourcc(*'XVID')
frame_height, frame_width = frame.shape[:2]
out = cv2.VideoWriter('tracked_output.avi', fourcc, 20.0, (frame_width, frame_height))

In [ ]:
cap = cv2.VideoCapture(0)
frame_number = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_number += 1
    timestamp = time.time()

    for tracker, oid in trackers:
        success, box = tracker.update(frame)
        if success:
            x, y, w, h = [int(v) for v in box]
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
            cv2.putText(frame, f"ID {oid}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            # Save to CSV
            with open(csv_file, mode='a', newline='') as file:
                writer = csv.writer(file)
                writer.writerow([oid, frame_number, timestamp, x, y, w, h])

    cv2.imshow("Camera", frame)
    out.write(frame)

    key = cv2.waitKey(1) & 0xFF
    if key == 27:  # ESC to exit
        break

In [ ]:
cap.release()
out.release()
cv2.destroyAllWindows()